In [ ]:
import mne
import numpy as np
import os
import glob
import matplotlib.pyplot as plt

# ============================================================
# 1. 定位 MOABB 下载的 GDF 文件
# ============================================================
# MOABB 默认下载到 ~/mne_data/MNE-bnci-data/database/data-sets/001-2014/
data_dir = os.path.expanduser("~/mne_data/MNE-bnci-data/database/data-sets/001-2014")

# 列出所有已下载的 GDF 文件
gdf_files = sorted(glob.glob(os.path.join(data_dir, "*.gdf")))
print("已下载的 GDF 文件:")
for f in gdf_files:
    print(f"  {os.path.basename(f)}")
print()

# 文件命名规则：
#   A01T.gdf ~ A09T.gdf  → 被试 1~9 的训练集（有标签）
#   A01E.gdf ~ A09E.gdf  → 被试 1~9 的测试集（标签在独立文件中）

# ============================================================
# 2. 用 read_raw_gdf 加载单个文件
# ============================================================
subject = 1
raw_fname = os.path.join(data_dir, f"A{subject:02d}T.gdf")

print(f"加载文件: {raw_fname}")
raw = mne.io.read_raw_gdf(raw_fname, preload=True)

print(f"\n{'='*50}")
print("Raw 对象基本信息")
print(f"{'='*50}")
print(f"采样率  : {raw.info['sfreq']} Hz")
print(f"通道数  : {len(raw.ch_names)}")
print(f"通道名  : {raw.ch_names}")
print(f"数据形状: {raw.get_data().shape}  (channels × samples)")
print(f"时长    : {raw.n_times / raw.info['sfreq']:.1f} 秒")

# ============================================================
# 3. 查看事件（Events）
# ============================================================
events = mne.find_events(raw, stim_channel='STI 014', shortest_event=0)
print(f"\n{'='*50}")
print("事件信息")
print(f"{'='*50}")
print(f"事件总数: {len(events)}")
print(f"事件 ID : {np.unique(events[:, 2])}")
print("\n前 10 个事件:")
print(events[:10])

# BNCI2014_001 (BCI Competition IV 2a) 事件编码：
#   769 → Left Hand (class 1)
#   770 → Right Hand (class 2)
#   771 → Feet (class 3)
#   772 → Tongue (class 4)
event_id = {
    'Left Hand':  769,
    'Right Hand': 770,
    'Feet':       771,
    'Tongue':     772,
}

# 过滤只保留 MI 事件
mi_events = events[events[:, 2].isin([769, 770, 771, 772])]
print(f"\nMI 事件数: {len(mi_events)}")
for name, eid in event_id.items():
    count = np.sum(mi_events[:, 2] == eid)
    print(f"  {name:12s} (ID={eid}): {count} trials")

# ============================================================
# 4. 带通滤波
# ============================================================
# 典型 mu/beta 频段: 4-38 Hz
raw.filter(l_freq=4.0, h_freq=38.0, fir_design='firwin')
print(f"\n已应用带通滤波: 4-38 Hz")

# ============================================================
# 5. 提取 Epoch
# ============================================================
tmin, tmax = 0.0, 4.0  # 事件后 0~4 秒

epochs = mne.Epochs(
    raw,
    events=mi_events,
    event_id=event_id,
    tmin=tmin,
    tmax=tmax,
    baseline=None,       # BCI 场景通常不做基线校正
    preload=True,
    reject_by_annotation=True,
)

print(f"\n{'='*50}")
print("Epoch 信息")
print(f"{'='*50}")
print(epochs)
print(f"Epoch 数据形状: {epochs.get_data().shape}")
print(f"  → (n_epochs, n_channels, n_samples)")

# ============================================================
# 6. 转为 NumPy 数组（方便后续机器学习）
# ============================================================
X = epochs.get_data()        # shape: (n_epochs, n_channels, n_samples)
y = epochs.events[:, 2]      # 事件 ID 作为标签

# 把事件 ID 映射为 0~3
label_map = {769: 0, 770: 1, 771: 2, 772: 3}
y_mapped = np.array([label_map[val] for val in y])
label_names = ['Left Hand', 'Right Hand', 'Feet', 'Tongue']

print(f"\n{'='*50}")
print("NumPy 数组")
print(f"{'='*50}")
print(f"X shape : {X.shape}")
print(f"y shape : {y_mapped.shape}")
for i, name in enumerate(label_names):
    print(f"  class {i} ({name:12s}): {np.sum(y_mapped == i)} samples")

# ============================================================
# 7. 可视化
# ============================================================
# 7a. 各类别的平均 ERP（取 C3、Cz、C4 三个关键通道）
picks = ['EEG-C3', 'EEG-Cz', 'EEG-C4']

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
colors = {'Left Hand': '#e74c3c', 'Right Hand': '#3498db',
          'Feet': '#2ecc71', 'Tongue': '#f39c12'}

for ax, ch in zip(axes, picks):
    for label_name, eid in event_id.items():
        evoked = epochs[label_name].average()
        ch_idx = evoked.ch_names.index(ch)
        ax.plot(evoked.times, evoked.data[ch_idx],
                label=label_name, color=colors[label_name], linewidth=1.2)
    ax.set_title(ch)
    ax.set_xlabel("Time (s)")
    ax.grid(True, alpha=0.3)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)

axes[0].set_ylabel("Amplitude (V)")
axes[-1].legend(loc='upper right', fontsize=8)
fig.suptitle(f"Subject {subject} — Average ERP", fontsize=14)
plt.tight_layout()
plt.savefig("erp_gdf.png", dpi=150)
plt.show()

# 7b. 画某次 trial 的原始信号
fig, ax = plt.subplots(figsize=(12, 4))
ch_picks = mne.pick_channels(raw.ch_names, ['EEG-C3', 'EEG-Cz', 'EEG-C4'])
t = epochs.times
for i, (idx, name) in enumerate(zip(ch_picks, picks)):
    ax.plot(t, X[0, idx, :] + i * 3e-5, label=name, linewidth=0.8)
ax.set_xlabel("Time (s)")
ax.set_title(f"Trial 0 — Label: {label_names[y_mapped[0]]}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("single_trial_gdf.png", dpi=150)
plt.show()

# ============================================================
# 8. 批量加载多个被试
# ============================================================
print(f"\n{'='*50}")
print("批量加载多个被试")
print(f"{'='*50}")

all_X, all_y, all_subject = [], [], []

for sid in range(1, 10):
    fname = os.path.join(data_dir, f"A{sid:02d}T.gdf")
    if not os.path.exists(fname):
        print(f"  Subject {sid}: 文件不存在，跳过")
        continue

    raw_i = mne.io.read_raw_gdf(fname, preload=True, verbose=False)
    raw_i.filter(l_freq=4.0, h_freq=38.0, fir_design='firwin', verbose=False)

    events_i = mne.find_events(raw_i, stim_channel='STI 014', shortest_event=0)
    mi_events_i = events_i[events_i[:, 2].isin([769, 770, 771, 772])]

    epochs_i = mne.Epochs(raw_i, mi_events_i, event_id,
                          tmin=0, tmax=4, baseline=None,
                          preload=True, verbose=False)

    X_i = epochs_i.get_data()
    y_i = np.array([label_map[e] for e in epochs_i.events[:, 2]])

    all_X.append(X_i)
    all_y.append(y_i)
    all_subject.append(np.full(len(y_i), sid))

    print(f"  Subject {sid}: {X_i.shape[0]} trials")

X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)
subject_all = np.concatenate(all_subject, axis=0)

print(f"\n全部数据: X={X_all.shape}, y={y_all.shape}")
